# 서브 워드 비교 실험

서브워드 라이브러리 HuggingFace Tokenizer와 SentencePiece를 비교하는 과제 노트북입니다.

진행 순서는 다음과 같습니다.

1. 데이터 준비 : 네이버 영화 리뷰 [[링크]](https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt)
2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정
3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석
4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정
5. 기타 추가 실험
6. 사용성과 장단점 분석

## 1. 데이터 준비 : 네이버 영화 리뷰

### HuggingFace Tokenizer와 SentencePiece 설치하기

In [26]:
!pip install -q sentencepiece tokenizers

### 네이버 리뷰 데이터 불러오기

In [27]:
import requests

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
response = requests.get(url)

with open("corpus.txt", "wb") as f:
    f.write(response.content)

In [28]:
import os

file_path = "/content/corpus.txt"

if os.path.exists(file_path):
    print("다운로드 완료")
else:
    print("파일을 찾을 수 없음")

다운로드 완료


### 데이터 정제하기
현재 데이터에는 `id`, `document`, `label`이 있습니다. 이 중 실질적인 문장 데이터인 `document`만 활용합니다.

In [29]:
import pandas as pd
import sentencepiece as spm

# corpus.txt는 id, document, label로 구성된 TSV 파일
df = pd.read_csv("corpus.txt", delimiter="\t")

# 결측치 제거
df = df.dropna(subset=["document"])
# SentencePiece 학습용 txt 파일 생성
with open("spm_input.txt", "w", encoding="utf-8") as f:
    for sentence in df["document"]:
        f.write(str(sentence).strip() + "\n")

## 2. Sentencepiece/HuggingFace를 사용하여 단어사전 구축 및 속도 측정

HuggingFace BPE는 SentencePiece와 내제된 철학이 다릅니다.  
고전적인 BPE는 단어 단위 알고리즘에서 출발하였으며 단어 목록을 입력으로 받아 문장 분리 -> 단어 분리 -> BPE를 가정합니다.  
때문에 이 철학을 따르는 HuggingFace BPE는 `Witespace()` 혹은 `ByteLevel()`과 같은 pre-tokenizer가 필요합니다.  
반면 SentencePiece는 공백도 학습하자는 철학으로 공백에 대하여 \_\_로 처리하여 기본 문장에서 공백을 기준으로 나누는 것은 동일하지만 문장 전체를 입력받게 되며 다음과 같이 처리합니다.  

"오늘은 날이 참 밝다."  
"오늘은\_\_날이\_\_참\_\_밝다."

만일 SentencePiece의 철학과 유사하게 진행하려면 `Metqaspace()`를 활용하십시오.

### Sentencepiece를 사용하여 단어사전 구축

In [52]:
import time
import sentencepiece as spm

start_time = time.perf_counter()

spm.SentencePieceTrainer.train(
    input="spm_input.txt",
    model_prefix="nsmc_unigram",
    vocab_size=1000,
    character_coverage=0.9995,
    model_type="unigram",
    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=3
)

sp_train_time = time.perf_counter() - start_time

print("SentencePiece Unigram 학습 완료")
print(f"SentencePiece 학습 시간: {sp_train_time:.4f}초")

SentencePiece Unigram 학습 완료
SentencePiece 학습 시간: 65.7631초


### HuggingFace Tokenizer를 사용하여 단어사전 구축

In [51]:
import time
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder

# HuggingFace BPE Tokenizer 생성
hf_tokenizer = Tokenizer(
    BPE(
        unk_token="<unk>"
    )
)

# SentencePiece처럼 공백을 ▁로 처리하기 위해 Metaspace 사용
# tokenizers 버전에 따라 인자가 다를 수 있으므로 try-except로 처리
try:
    hf_tokenizer.pre_tokenizer = Metaspace(
        replacement="▁",
        prepend_scheme="always"
    )
    hf_tokenizer.decoder = MetaspaceDecoder(
        replacement="▁",
        prepend_scheme="always"
    )
except TypeError:
    hf_tokenizer.pre_tokenizer = Metaspace(
        replacement="▁",
        add_prefix_space=True
    )
    hf_tokenizer.decoder = MetaspaceDecoder(
        replacement="▁",
        add_prefix_space=True
    )

# BPE Trainer 설정
hf_trainer = BpeTrainer(
    vocab_size=8000,
    special_tokens=["<unk>", "<s>", "</s>", "<pad>"]
)

# HuggingFace BPE 학습 속도 측정
start_time = time.perf_counter()

hf_tokenizer.train(
    files=["spm_input.txt"],
    trainer=hf_trainer
)

hf_train_time = time.perf_counter() - start_time

# 학습된 tokenizer 저장
hf_tokenizer.save("nsmc_hf_bpe.json")

print("HuggingFace BPE 학습 완료")
print(f"HuggingFace BPE 학습 시간: {hf_train_time:.4f}초")
print("생성된 파일: nsmc_hf_bpe.json")

HuggingFace BPE 학습 완료
HuggingFace BPE 학습 시간: 7.8232초
생성된 파일: nsmc_hf_bpe.json


## 3. 1과 2 결과물의 상위 빈도 30개의 서브워드 분석

### 상위 30개 기준, 단어사전 조회하기

In [46]:
import pandas as pd

# =========================
# 1. SentencePiece vocab 상위 30개 조회
# =========================

sp_vocab_list = []

with open("nsmc_unigram.vocab", "r", encoding="utf-8") as f:
    for line in f:
        token, score = line.strip().split("\t")
        sp_vocab_list.append((token, score))

sp_vocab_top30 = pd.DataFrame(
    sp_vocab_list[:30],
    columns=["sentencepiece_token", "score"]
)

print("SentencePiece vocab 상위 30개")
display(sp_vocab_top30)


# =========================
# 2. HuggingFace BPE vocab 상위 30개 조회
# =========================

hf_vocab = hf_tokenizer.get_vocab()

# get_vocab()은 {token: id} 형태이므로 id 기준으로 정렬
hf_vocab_sorted = sorted(
    hf_vocab.items(),
    key=lambda x: x[1]
)

hf_vocab_top30 = pd.DataFrame(
    hf_vocab_sorted[:30],
    columns=["huggingface_bpe_token", "id"]
)

print("HuggingFace BPE vocab 상위 30개")
display(hf_vocab_top30)

SentencePiece vocab 상위 30개


,sentencepiece_token,score
0,<unk>,0
1,<s>,0
2,</s>,0
3,▁,-3.23267
4,.,-3.44376
5,..,-4.35967
6,이,-4.42178
7,▁영화,-4.55895
8,...,-4.60112
9,의,-4.68391


HuggingFace BPE vocab 상위 30개


,huggingface_bpe_token,id
0,<unk>,0
1,<s>,1
2,</s>,2
3,<pad>,3
4,\n,4
5,!,5
6,"""",6
7,#,7
8,$,8
9,%,9


### 분절된 토큰 빈도 조회

In [47]:
from collections import Counter
import pandas as pd
import sentencepiece as spm

# =========================
# 0. 문장 리스트 준비
# =========================

sentences = df["document"].dropna().astype(str).str.strip().tolist()
sentences = [s for s in sentences if s != ""]

print("분석 문장 수:", len(sentences))


# =========================
# 1. SentencePiece 모델 로드
# =========================

sp = spm.SentencePieceProcessor()
sp.load("nsmc_unigram.model")


# =========================
# 2. SentencePiece 분절 토큰 빈도 계산
# =========================

sp_token_counter = Counter()

for sentence in sentences:
    tokens = sp.encode_as_pieces(sentence)
    sp_token_counter.update(tokens)

sp_token_freq_top30 = pd.DataFrame(
    sp_token_counter.most_common(30),
    columns=["sentencepiece_token", "frequency"]
)

print("SentencePiece 분절 토큰 빈도 상위 30개")
display(sp_token_freq_top30)


# =========================
# 3. HuggingFace BPE 분절 토큰 빈도 계산
# =========================

hf_token_counter = Counter()

batch_size = 1000

for i in range(0, len(sentences), batch_size):
    batch = sentences[i:i + batch_size]
    encodings = hf_tokenizer.encode_batch(batch)

    for encoding in encodings:
        hf_token_counter.update(encoding.tokens)

hf_token_freq_top30 = pd.DataFrame(
    hf_token_counter.most_common(30),
    columns=["huggingface_bpe_token", "frequency"]
)

print("HuggingFace BPE 분절 토큰 빈도 상위 30개")
display(hf_token_freq_top30)

분석 문장 수: 149995
SentencePiece 분절 토큰 빈도 상위 30개


,sentencepiece_token,frequency
0,.,69811
1,▁,65668
2,..,30957
3,이,26639
4,...,25431
5,▁영화,24264
6,의,22025
7,가,21364
8,도,20131
9,",",18172


HuggingFace BPE 분절 토큰 빈도 상위 30개


,huggingface_bpe_token,frequency
0,▁,1137736
1,.,241461
2,이,137121
3,다,110236
4,는,90570
5,고,70831
6,지,66721
7,화,65575
8,영,64734
9,가,57022


## 4. 학습 속도(단어사전 구축 속도), 단일 문장 변환 속도, 코퍼스 변환 속도 차이 특정

### 단일 문장 변환 속도 측정

In [42]:
import time

test_sentence = "이 영화는 정말 재미있고 배우들의 연기도 좋았다."

repeat = 10000

# =========================
# 1. SentencePiece 단일 문장 변환 속도
# =========================

start_time = time.perf_counter()

for _ in range(repeat):
    sp.encode_as_ids(test_sentence)

sp_single_time = time.perf_counter() - start_time
sp_single_avg = sp_single_time / repeat


# =========================
# 2. HuggingFace BPE 단일 문장 변환 속도
# =========================

start_time = time.perf_counter()

for _ in range(repeat):
    hf_tokenizer.encode(test_sentence).ids

hf_single_time = time.perf_counter() - start_time
hf_single_avg = hf_single_time / repeat


print("단일 문장 변환 속도 측정 결과")
print(f"반복 횟수: {repeat}")
print(f"SentencePiece 전체 시간: {sp_single_time:.6f}초")
print(f"SentencePiece 평균 시간: {sp_single_avg * 1000:.6f} ms")
print(f"HuggingFace BPE 전체 시간: {hf_single_time:.6f}초")
print(f"HuggingFace BPE 평균 시간: {hf_single_avg * 1000:.6f} ms")

단일 문장 변환 속도 측정 결과
반복 횟수: 10000
SentencePiece 전체 시간: 0.078505초
SentencePiece 평균 시간: 0.007850 ms
HuggingFace BPE 전체 시간: 0.218734초
HuggingFace BPE 평균 시간: 0.021873 ms


### 코퍼스 전체 변환 속도 측정

In [35]:
# =========================
# 1. SentencePiece 코퍼스 전체 변환 속도
# =========================

start_time = time.perf_counter()

sp_encoded_corpus = [
    sp.encode_as_ids(sentence)
    for sentence in sentences
]

sp_corpus_time = time.perf_counter() - start_time


# =========================
# 2. HuggingFace BPE 코퍼스 전체 변환 속도
# =========================

start_time = time.perf_counter()

hf_encoded_corpus = []

batch_size = 1000

for i in range(0, len(sentences), batch_size):
    batch = sentences[i:i + batch_size]
    encodings = hf_tokenizer.encode_batch(batch)
    hf_encoded_corpus.extend([encoding.ids for encoding in encodings])

hf_corpus_time = time.perf_counter() - start_time


print("코퍼스 전체 변환 속도 측정 결과")
print(f"문장 수: {len(sentences)}")
print(f"SentencePiece 전체 변환 시간: {sp_corpus_time:.6f}초")
print(f"HuggingFace BPE 전체 변환 시간: {hf_corpus_time:.6f}초")

print()
print(f"SentencePiece 문장당 평균 시간: {sp_corpus_time / len(sentences) * 1000:.6f} ms")
print(f"HuggingFace BPE 문장당 평균 시간: {hf_corpus_time / len(sentences) * 1000:.6f} ms")

코퍼스 전체 변환 속도 측정 결과
문장 수: 149995
SentencePiece 전체 변환 시간: 2.369724초
HuggingFace BPE 전체 변환 시간: 5.561662초

SentencePiece 문장당 평균 시간: 0.015799 ms
HuggingFace BPE 문장당 평균 시간: 0.037079 ms


## 5. 기타 추가 실험

In [36]:
# [[YOUR CODE]]

## 6. 사용성과 장단점 분석

**장점**  

**단점**